# T_test2 + strong baselines — typed verification

Ready to run on Colab (GPU T4). It reuses, **verbatim**, the frozen compiler / router / branches (`code/frozen_compiler.py`) and the T_test
bank of the companion manuscript. Nothing in the verifier is re-implemented.

**What it adds**
1. **T_test2**, a second held-out phrasing set (27 templates × 300 *new* flows), on which we
   evaluate **V0** (the companion manuscript's verifier, frozen) and **V1**, which differs by the single structural
   rule announced before T_test2 was written: *an operator-free feature mention (compiled `RANGE`) is routed to
   abstention instead of EXEC*. No lexicon entry is added.
2. **Stronger baselines on the same claims**: MiniCheck RoBERTa-L and Flan-T5-L, plus the NLI
   cross-encoder of the companion manuscript; paired McNemar tests and two-way cluster bootstrap.
3. Per-claim cost, and the real-rationale refutation rate under V0 and V1.

**Guards**

| Section | Guard |
|---|---|
| §1 | The compiler, T_test bank and sentence helpers are stored as text and checked by SHA-256 before `exec`: any edit stops the run. |
| §2 | T_test is replayed exactly (same flows, same templates). It must give 0.864 end-to-end, 0.833 routing, 489 errors, 218 false SUPPORTED; otherwise the run stops. |
| §3 | T_test2 + the V1 rule are hashed and written to Drive **before** any verifier sees a T_test2 claim. A later edit is refused. |
| §3 | *Neutral* templates (written from the grammar) carry the generalisation claim; *probe* templates (aimed at patterns P1–P3) are a stress test, reported separately; *external* templates (optional) are written blind by a co-author **before the first run**. |
| §5 | One claim per (template, flow) cell; two-way cluster bootstrap; exact McNemar; one-sided Clopper–Pearson. |

T_test keeps its role: V1 on T_test is **diagnostic only** (the rule was motivated by it).

**Run:** Runtime → GPU, then Run all (≈ 40–60 min, mostly MiniCheck). Results go to
`/content/drive/MyDrive/project/ttest2/`.

In [ ]:
# === 0. Setup =================================================================
import os, sys, re, json, time, hashlib, datetime, subprocess, importlib, warnings
os.environ["TQDM_DISABLE"] = "1"                   # no per-claim progress bars (they overload Colab)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
DRYRUN = os.environ.get("TTEST2_DRYRUN") == "1"        # authors' plumbing test only (synthetic data)

class CFG:
    MODELS_ROOT = "/content/drive/MyDrive/models"
    TINY_DIR   = "/content/drive/MyDrive/models/TinyLlama_LoRA_r16_3ep_CICIoT2023"
    OUTPUT_DIR = "/content/drive/MyDrive/project/article_outputs"
    WORK       = "/content/drive/MyDrive/project/ttest2"     # results (persistent)
    DATASET    = "CICIoT2023"
    NLI_MODEL  = "cross-encoder/nli-deberta-v3-small"
    SEED       = 42          # T_dev / T_test flows (as in the companion manuscript)
    SEED2      = 2027        # T_test2 flows and feature draws
    B_BOOT     = 10_000
    MINICHECK  = ["roberta-large", "flan-t5-large"]      # add "deberta-v3-large" if time allows
    BASELINE_SETS = ["ttest2", "ttest"]                  # MiniCheck/NLI on both held-out sets
    DRIVE_ROOT = "/content/drive/MyDrive"
    # Qwen2.5 rationales (second generator): "auto" = found on Drive automatically
    # (any CSV whose name contains 'qwen', with a flow-index column and a text column).
    QWEN_CSV   = "auto"
    OUT        = "results"
    STRICT_REPRO = True      # stop if T_test is not reproduced; set False ONLY to inspect a mismatch
STRICT_REPRO = CFG.STRICT_REPRO

if DRYRUN:
    CFG.WORK = os.path.abspath(os.environ.get("TTEST2_WORK", "ttest2_out"))
    CFG.OUTPUT_DIR = CFG.DRIVE_ROOT = CFG.WORK
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Not on Colab or Drive not mounted:", e)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main",
                    "sentence-transformers", "joblib", "scikit-learn"], check=False)
    import nltk
    for pkg in ("punkt", "punkt_tab"):          # MiniCheck sentence-splits the documents
        nltk.download(pkg, quiet=True)
os.makedirs(CFG.WORK, exist_ok=True); os.makedirs(f"{CFG.OUT}/tables", exist_ok=True)
import numpy as np, pandas as pd
from scipy.stats import beta, binomtest
def tag(kind, msg): print(f"[{kind}] {msg}")
tag("SETUP", f"results -> {CFG.WORK} | dryrun={DRYRUN}")

In [ ]:
# === Load the real artifacts (same code path as the companion manuscript, no demo fallback) ===
def load_live():
    import joblib
    d = CFG.TINY_DIR
    if not os.path.exists(os.path.join(d, "label_encoder.joblib")):
        import glob
        hits = [h for h in glob.glob(CFG.DRIVE_ROOT + "/**/label_encoder.joblib", recursive=True)
                if "TinyLlama" in h and CFG.DATASET in h]
        if not hits:
            raise FileNotFoundError("label_encoder.joblib not found: set CFG.TINY_DIR")
        d = CFG.TINY_DIR = os.path.dirname(hits[0]); tag("AUTO", f"TINY_DIR = {d}")
    le     = joblib.load(os.path.join(d, "label_encoder.joblib"))
    classes= list(le.classes_)
    feats  = list(joblib.load(os.path.join(d, "top_features.joblib")))
    scaler = joblib.load(os.path.join(d, "scaler.joblib"))
    Xsc    = np.load(os.path.join(d, "X_test_sc.npy"))
    Xraw   = scaler.inverse_transform(Xsc)
    y      = np.load(os.path.join(d, "y_test.npy"))
    return classes, feats, Xraw, y

def load_dryrun():   # synthetic, ONLY for the authors' plumbing test (TTEST2_DRYRUN=1)
    rng = np.random.RandomState(0)
    feats = ["flow_duration", "Header_Length", "Protocol Type", "Duration", "Rate", "Srate", "Tot sum",
             "Min", "Max", "AVG", "Std", "Tot size", "IAT", "Magnitue", "Radius", "Covariance",
             "Variance", "Weight", "ack_count", "syn_count", "fin_count", "urg_count", "rst_count",
             "Drate", "Tot pkts"]
    classes = ["Benign", "DDoS", "DoS", "Mirai", "Recon", "Spoofing"]
    Xraw = np.abs(rng.lognormal(1.0, 1.2, size=(1800, len(feats)))); Xraw[rng.rand(*Xraw.shape) < .08] = 0
    return classes, feats, Xraw, rng.randint(0, len(classes), 1800)

classes, feats, Xraw, y = load_dryrun() if DRYRUN else load_live()
y_str = np.array([classes[int(i)] for i in y], dtype=object)
feat_q = {}
for i, f in enumerate(feats):
    col = Xraw[:, i].astype(float)
    feat_q[f] = {"q33": float(np.percentile(col, 33)), "q66": float(np.percentile(col, 66)),
                 "med": float(np.median(col))}
ATTACK = [c for c in classes if c.lower() != "benign"]
tag("DATA", f"{Xraw.shape[0]} flows x {len(feats)} features | classes={classes}")

## 1. Frozen verifier (verbatim, hash-checked)

In [ ]:
FROZEN_COMPILER = r'''# === Claim compiler, router, and 4-verdict verifier =========================
import re
GRAMMAR=["VALUE","RANGE","COMPARISON","QUANTIFIER","ABSENCE","CLASS","BEHAVIOR","CAUSALITY"]
LOW={"low","short","small","brief","minimal","light"}; HIGH={"high","long","large","heavy","elevated","big"}
MED={"medium","moderate","average","typical"}; FEW={"few","only a few","little","fewer"}
# vague / underspecified magnitudes: no executable predicate -> must NOT be SUPPORTED
VAGUE={"unusual","atypical","abnormal","strange","irregular","notable","remarkable",
       "significant","considerable","relatively unusual","somewhat unusual","suspicious"}
MANY={"many","lots","numerous","sustained","high-volume","high volume","large number"}
BEHAV_NAMES={"reconnaissance","recon","exfiltration","beaconing","command-and-control","c2",
       "brute-force","bruteforce","scanning","port scan","portscan","probing","lateral movement",
       "denial of service","dos","ddos","volumetric","flooding","flood","spoofing","injection",
       "malware","botnet","mirai"}
BEHAV_CUES={"consistent with","indicative of","characteristic of","typical of","suggests",
       "pattern of","resembles","looks like","compatible with"}
BEHAV=BEHAV_NAMES|BEHAV_CUES
BENIGN={"benign","normal","legitimate","harmless","expected","safe"}
# keyword-independent causal / abusive-conclusion cues (explicit, implicit, inverted, conclusion)
CAUSAL_EXPLICIT={"caused","because","therefore","thus","hence","results in","leads to","due to"}
CAUSAL_IMPLICIT={"as a consequence","consequence of","resulted from","gives rise","stems from",
                 "attributable to","on account of","owing to"}
INTENT={"intent","intentional","deliberately","in order to","so as to","evade","evasion","attempts to"}
CONCLUSION={"is established","is confirmed","is certainly","certainly malicious","must be",
            "proves","demonstrates that","it follows that","can only mean"}
def real_values(j): return {feats[i]: float(Xraw[j][i]) for i in range(len(feats))}
def _has(words,c): return any(re.search(r"(?<![a-z])"+re.escape(w)+r"(?![a-z])",c) for w in words)
def _num(s):
    m=re.search(r"[-+]?\d+\.?\d*",s); return float(m.group()) if m else None
# --- feature synonym resolution: generated text rarely uses the exact schema name ---
SYN_TOKENS={"pkt":["packet","packets"],"len":["length","size"],"dur":["duration","time","lasted"],
            "fwd":["forward","outbound","upstream"],"bwd":["backward","inbound","downstream","response"],
            "iat":["inter-arrival","interarrival","arrival gap"],"tot":["total"],
            "byts":["bytes","volume"],"cnt":["count","number"],"avg":["average","mean"],
            "min":["minimum","smallest"],"max":["maximum","largest"],"std":["deviation","variance"],
            "rate":["rate","per second"],"flag":["flag","flags"],"hdr":["header"],"seg":["segment"]}
def _tok_syn(f):
    parts=f.lower().replace('_',' ').split()
    return [set([p])|set(SYN_TOKENS.get(p,[])) for p in parts]
_FEAT_TOKENS={f:_tok_syn(f) for f in feats}
def _feat(c):
    # score each schema feature by how many of its (synonym-expanded) tokens appear.
    # Requires a majority of tokens AND >=2 matches; ties -> None (ambiguous, conservative).
    best=[]; bestn=0
    for f in feats:
        toks=_FEAT_TOKENS[f]
        n=sum(1 for syns in toks if any(re.search(r"(?<![a-z])"+re.escape(w)+r"(?![a-z])",c) for w in syns))
        need=max(2, (len(toks)+1)//2) if len(toks)>1 else 1
        if n>=need:
            if n>bestn: best=[f]; bestn=n
            elif n==bestn: best.append(f)
    return best[0] if len(best)==1 else None
# a term that *looks like* a network feature but resolves to nothing in the schema
FEATUREISH={"entropy","handshake","tls","ssl","jitter","latency","throughput","payload",
            "checksum","ttl","window","retransmission","fragment","certificate","cipher"}
def is_causal_or_abusive(c):
    # structural: any causal/intent cue, OR a compound claim ending in an unwarranted conclusion
    if _has(CAUSAL_EXPLICIT,c) or _has(CAUSAL_IMPLICIT,c) or _has(INTENT,c) or _has(CONCLUSION,c):
        return True
    # conclusion pattern: "..., <claim>" with a malicious/benign verdict but no evidence operator
    if re.search(r"(malicious|attack|benign|normal)\b", c) and (";" in c or "," in c) and \
       not _feat(c): return True
    return False
def compile_claim(claim):
    c=claim.lower().strip()
    if is_causal_or_abusive(c):        return {"category":"CAUSALITY"}
    # CLASS first: "classified as Recon" / "predicts DDoS" is a decision restatement, not a
    # behavioral compatibility claim (previously mis-routed to CONSTRAINT on the keyword).
    _CLASS_NAMES={str(k).lower() for k in classes}
    _names_in = any(re.search(r"(?<![a-z])"+re.escape(k)+r"(?![a-z])", c) for k in _CLASS_NAMES)
    if (_has(BENIGN,c) or "classifier predicts" in c or "classified as" in c
        or "this is a" in c or "flow classified" in c or "labelled as" in c
        or "labeled as" in c): return {"category":"CLASS"}
    # decision restatements: "this flow is (likely) a DDoS attack", "it is a DoS", "indicates
    # a DoS attack" -- these assert the LABEL, not behavioural compatibility. Route to CLASS.
    if re.search(r"(is|are|be|was)\s+(indeed\s+|likely\s+|probably\s+|clearly\s+)?(an?\s+)?"
                 r"[a-z0-9\-]*\s*(attack|flow|traffic|activity)?", c) and (_names_in or _has(BEHAV_NAMES,c)) \
       and re.search(r"(this|the)\s+(flow|traffic|session|connection|sample)", c):
        return {"category":"CLASS"}
    if _names_in and not _has(BEHAV_CUES,c):            # bare class name, no compatibility cue
        return {"category":"CLASS"}
    if _has(BEHAV_CUES,c) and _has(BEHAV_NAMES,c):   # needs a compatibility cue AND a behavior name
        return {"category":"BEHAVIOR","behavior":next(w for w in BEHAV_NAMES if _has({w},c))}
    if _has(BEHAV_NAMES,c):            return {"category":"BEHAVIOR","behavior":next(w for w in BEHAV_NAMES if _has({w},c))}
    f=_feat(c)
    if f and _has({"no","zero","absent"},c):                 return {"category":"ABSENCE","feature":f}
    if f and _has(FEW|MANY,c):                               return {"category":"QUANTIFIER","feature":f,"dir":"few" if _has(FEW,c) else "many"}
    if f and _num(c) is not None and not _has(LOW|HIGH|MED,c):return {"category":"VALUE","feature":f,"operand":_num(c)}
    if f and _has(VAGUE,c) and not _has(LOW|HIGH|MED,c):
        return {"category":"VAGUE","feature":f}              # underspecified -> not executable
    if f and _has(LOW|HIGH|MED,c):
        d="low" if _has(LOW,c) else ("high" if _has(HIGH,c) else "medium")
        return {"category":"COMPARISON","feature":f,"dir":d}
    if f:                                                    return {"category":"RANGE","feature":f}
    return {"category":"CAUSALITY"}
def route(cat): return "EXEC" if cat in {"VALUE","RANGE","COMPARISON","QUANTIFIER","ABSENCE","CLASS"} \
                       else ("CONSTRAINT" if cat=="BEHAVIOR" else "SEMANTIC")   # VAGUE/CAUSALITY -> SEMANTIC -> abstain
def _band(f,v):
    q=feat_q[f]; return "low" if v<q["q33"] else ("high" if v>q["q66"] else "medium")
TOL=0.02
# Behaviour -> observable constraints. Each mapping states, in terms of the two flow
# observables we actually have (packet volume, duration), what the named behaviour would
# LOOK like. These are compatibility conditions, never proofs: full or partial satisfaction
# yields CONSISTENT, never SUPPORTED. Every name in BEHAV_NAMES must have an entry, or the
# branch silently returns UNRESOLVED and CONSISTENT can never fire.
BCON={
 # probing / enumeration: little data, short exchanges
 "reconnaissance":["few","short"], "recon":["few","short"],
 "scanning":["few","short"], "port scan":["few","short"], "portscan":["few","short"],
 "probing":["few","short"],
 # flooding / denial of service: high volume is the defining observable
 "denial of service":["many"], "dos":["many"], "ddos":["many"],
 "volumetric":["many"], "flooding":["many"], "flood":["many"],
 # bulk transfer out: high volume, sustained
 "exfiltration":["many","long"],
 # periodic low-rate callbacks
 "beaconing":["few"], "command-and-control":["few"], "c2":["few"],
 # repeated attempts: many small exchanges
 "brute-force":["many","short"], "bruteforce":["many","short"],
 # remaining families have no distinctive volume/duration signature in our feature set,
 # so we declare no constraint: the branch must abstain rather than invent one.
 "lateral movement":[], "spoofing":[], "injection":[], "malware":[], "botnet":[], "mirai":[],
}
_MISSING_BCON = {b for b in BEHAV_NAMES if b not in BCON}
assert not _MISSING_BCON, f"behaviour without a declared constraint set: {_MISSING_BCON}"
def _pf():
    for f in feats:
        if "pkt" in f.lower() or "packet" in f.lower(): return f
    return feats[0]
def _df():
    for f in feats:
        if "dur" in f.lower(): return f
    return feats[1] if len(feats)>1 else feats[0]
def verify(claim,j,pred=None):
    pred=pred if pred is not None else y_str[j]; rv=real_values(j)
    comp=compile_claim(claim); br=route(comp["category"]); cat=comp["category"]
    if br=="EXEC":
        if cat=="VALUE":
            f=comp["feature"]; return ("SUPPORTED" if abs(rv[f]-comp["operand"])<=TOL*max(abs(rv[f]),1e-6) else "CONTRADICTED"),br,cat
        if cat in ("COMPARISON","RANGE"):
            f=comp["feature"]; return ("SUPPORTED" if _band(f,rv[f])==comp.get("dir","medium") else "CONTRADICTED"),br,cat
        if cat=="QUANTIFIER":
            f=comp["feature"]; b=_band(f,rv[f]); w=comp["dir"]
            if (w=="few" and b=="low") or (w=="many" and b=="high"): return "SUPPORTED",br,cat
            if (w=="few" and b=="high") or (w=="many" and b=="low"): return "CONTRADICTED",br,cat
            return "UNRESOLVED",br,cat
        if cat=="ABSENCE":
            f=comp["feature"]; return ("SUPPORTED" if abs(rv[f])<1e-9 else "CONTRADICTED"),br,cat
        if cat=="CLASS":
            # Compare the class the claim NAMES against the decision, instead of assuming
            # benign wording: "this flow is a DDoS attack" restates an attack decision
            # correctly, while benign wording on an attack decision contradicts it.
            cl=claim.lower()
            named=[k for k in classes
                   if re.search(r"(?<![a-z])"+re.escape(str(k).lower())+r"(?![a-z])", cl)]
            if named:
                return ("SUPPORTED" if any(str(k).lower()==str(pred).lower() for k in named)
                        else "CONTRADICTED"), br, cat
            if _has(BENIGN,cl):                 # benign wording, no explicit class name
                return ("CONTRADICTED" if pred in ATTACK else "SUPPORTED"),br,cat
            return "UNRESOLVED",br,cat          # a class statement naming nothing checkable
    if br=="CONSTRAINT":
        cons=BCON.get(comp.get("behavior",""),[])
        if not cons: return "UNRESOLVED",br,cat      # no declared observable signature -> abstain
        pf,df=_pf(),_df(); sat=0
        for k in cons:
            if k=="few" and _band(pf,rv[pf])=="low": sat+=1
            elif k=="many" and _band(pf,rv[pf])=="high": sat+=1
            elif k=="medium" and _band(pf,rv[pf])=="medium": sat+=1
            elif k=="short" and _band(df,rv[df])=="low": sat+=1
            elif k=="long" and _band(df,rv[df])=="high": sat+=1
        # A behaviour is never *entailed* by one flow, so full satisfaction is still only
        # CONSISTENT. Partial satisfaction is also CONSISTENT (evidence compatible, not
        # probative); zero satisfaction contradicts the stated behaviour.
        score=sat/len(cons)
        if score==0.0:  return "CONTRADICTED",br,cat
        if score>=0.5:  return "CONSISTENT",br,cat
        return "UNRESOLVED",br,cat
    return "UNRESOLVED",br,cat            # SEMANTIC/CAUSALITY -> abstain
for ex in ["The traffic is benign and normal.",
           "These statistics indicate reconnaissance.",
           "The attack occurred as a consequence of the limited packet exchange.",
           "Given the short flow and low packet count, malicious intent is established."]:
    print(f"  {ex[:52]:54s} -> {verify(ex,int(0))}")
tag("VERIFIER","4 verdicts {SUPPORTED,CONTRADICTED,CONSISTENT,UNRESOLVED}; causality keyword-independent")'''
assert hashlib.sha256(FROZEN_COMPILER.encode()).hexdigest() == "83ccacc6ca15b6fc50dee17d07154e44f7bd81807cb25d827e1133ef9aa2a7c8", "compiler text differs from the frozen compiler: STOP"
exec(FROZEN_COMPILER)
tag("FROZEN", "compiler/router/branches = frozen compiler (sha256 ok)")

In [ ]:
FROZEN_HELPERS = r'''# Discourse / meta sentences are not claims about the flow: they state what one *would*
# do, ask a question, or introduce the answer. Auditing them produces false flags.
META_CUES={"we need to consider","to make a decision","in terms of","let us","let's",
           "in order to determine","the following","as follows","in summary","to summarize",
           "based on the above","here is","i will","we can use","we should","it is important",
           "note that","for example","in this case we","the goal is","first,","second,",
           "step 1","step 2","explanation:","answer:","justification:"}
def is_meta(s):
    c=s.lower().strip()
    if c.endswith("?"): return True                      # a question asserts nothing
    if _has(META_CUES,c) or any(k in c for k in META_CUES): return True
    if re.match(r"^\s*(to|in order to|how|why|what|when)\b", c): return True
    return False

def split_sentences(text):
    # Generator output is not reliably punctuated: a trailing sentence without a final
    # period would otherwise be merged with the next one, so the audit would analyse a
    # composite string and silently miss a claim. Split on terminal punctuation AND on
    # newlines / bullet markers, then re-split overlong fragments on " The <feature>".
    t=re.sub(r"\s*[\r\n]+\s*", " . ", str(text).strip())
    t=re.sub(r"\s+[-*\u2022]\s+", " . ", t)
    parts=re.split(r"(?<=[.!?])\s+", t)
    out=[]
    for p in parts:
        p=p.strip()
        if not p: continue
        # a fragment holding several "The <something> is ..." assertions -> split them
        if len(p.split())>25 and len(re.findall(r"(?i)\bthe\s+[a-z_ ]{2,30}\s+is\b", p))>1:
            sub=re.split(r"(?i)(?=\bthe\s+[a-z_ ]{2,30}\s+is\b)", p)
            out+= [x.strip() for x in sub if len(x.split())>=3]
        else:
            out.append(p)
    return [x.strip(" .") for x in out if len(x.split())>=3]

'''
assert hashlib.sha256(FROZEN_HELPERS.encode()).hexdigest() == "4b24155ec040964b884dad9bd5a39e3da7c88e68d27073803b36ac78e9643a50"
exec(FROZEN_HELPERS)
tag("FROZEN", "is_meta / split_sentences = plan_verify_generate_audit cell 10 (sha256 ok)")

In [ ]:
# The verifiers under test --------------------------------------------------------
EXEC_CATS = {"VALUE", "RANGE", "COMPARISON", "QUANTIFIER", "ABSENCE", "CLASS"}
GROUP = {"VALUE": "VAL", "RANGE": "CMP", "COMPARISON": "CMP", "QUANTIFIER": "QNT",
         "ABSENCE": "ABN", "CLASS": "CLS", "BEHAVIOR": "BEH"}
EXEC_GROUPS = {"VAL", "CMP", "QNT", "ABN", "CLS"}
def group_of(cat): return GROUP.get(cat, "UND")          # UND = abstention (CAUSALITY, VAGUE, ...)

def V0(text, j, pred=None):
    v, br, cat = verify(text, j, pred)
    return v, br, cat

# V1: the single structural change announced before T_test2 was written. RANGE is, by the
# compiler's own construction (last line of compile_claim), the type of a feature mention
# with no recognised operator; V1 sends it to abstention. Everything else is V0.
V1_RULE = "compile_claim(...)['category'] == 'RANGE'  ->  SEMANTIC / UNRESOLVED"
def V1_from_V0(v, br, cat):
    if cat == "RANGE":
        return "UNRESOLVED", "SEMANTIC", "OPFREE"
    return v, br, cat

# Gold for behavioural claims, computed from the DECLARED conditions (BCON) directly,
# without compile_claim, exactly as the CONSTRAINT branch would score a correctly
# compiled claim (approval invariant).
def bcon_gold(name, j):
    rv = real_values(j); cons = BCON.get(name, [])
    if not cons: return "UNRESOLVED"
    pf, df_ = _pf(), _df(); sat = 0
    for k in cons:
        if   k == "few"    and _band(pf, rv[pf]) == "low":     sat += 1
        elif k == "many"   and _band(pf, rv[pf]) == "high":    sat += 1
        elif k == "medium" and _band(pf, rv[pf]) == "medium":  sat += 1
        elif k == "short"  and _band(df_, rv[df_]) == "low":   sat += 1
        elif k == "long"   and _band(df_, rv[df_]) == "high":  sat += 1
    s = sat / len(cons)
    return "CONTRADICTED" if s == 0 else ("CONSISTENT" if s >= .5 else "UNRESOLVED")

## 2. Reproduction of T_test (companion manuscript)

In [ ]:
FROZEN_BANK = r'''# === Template banks: T_dev (seen) and T_test (UNSEEN, disjoint) =============
# Each entry: (category, builder(j) -> (text, gold_verdict))
def _f0(): return feats[0]
def _f1(): return feats[1] if len(feats)>1 else feats[0]
def _nm(f): return f.replace('_',' ')
INV={"low":"high","high":"low","medium":"medium"}

T_dev=[
 ("COMPARISON", lambda j: (f"The {_nm(_f0())} is {_band(_f0(),real_values(j)[_f0()])} for this dataset.","SUPPORTED")),
 ("COMPARISON", lambda j: (f"The {_nm(_f0())} is {INV[_band(_f0(),real_values(j)[_f0()])]} for this dataset.",
                           "CONTRADICTED" if _band(_f0(),real_values(j)[_f0()])!="medium" else "SUPPORTED")),
 ("VALUE",      lambda j: (f"The {_nm(_f0())} is {round(real_values(j)[_f0()],3)}.","SUPPORTED")),
 ("QUANTIFIER", lambda j: (f"The flow shows few {_nm(_f1())}.",
                           {"low":"SUPPORTED","high":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 ("CLASS",      lambda j: (f"The classifier predicts {y_str[j]}.","SUPPORTED")),
 ("BEHAVIOR",   lambda j: ("These statistics indicate reconnaissance.", None)),   # gold = CONSISTENT/CONTRA per flow
 ("CAUSALITY",  lambda j: (f"The low {_nm(_f1())} caused the observed attack.","UNRESOLVED")),
]

# --- UNSEEN paraphrases: different syntax and lexicon, same semantics -------------
T_test=[
 # COMPARISON: hedged, inverted, comparative-clause phrasings
 ("COMPARISON", lambda j: (f"Compared with the rest of the corpus, the {_nm(_f0())} sits in the "
                           f"{_band(_f0(),real_values(j)[_f0()])} band.","SUPPORTED")),
 ("COMPARISON", lambda j: (f"Relative to the dataset, this sample's {_nm(_f0())} registers as "
                           f"{INV[_band(_f0(),real_values(j)[_f0()])]}.",
                           "CONTRADICTED" if _band(_f0(),real_values(j)[_f0()])!="medium" else "SUPPORTED")),
 # VALUE: passive / measurement phrasing
 ("VALUE",      lambda j: (f"A {_nm(_f0())} of {round(real_values(j)[_f0()],3)} was recorded for this flow.","SUPPORTED")),
 ("VALUE",      lambda j: (f"This flow's {_nm(_f0())} was measured at {round(real_values(j)[_f0()]*3.1+7.0,3)}.","CONTRADICTED")),
 # QUANTIFIER: unseen quantifier lexicon
 ("QUANTIFIER", lambda j: (f"Only a handful of {_nm(_f1())} appear in this exchange.",
                           {"low":"SUPPORTED","high":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 ("QUANTIFIER", lambda j: (f"This exchange carries numerous {_nm(_f1())}.",
                           {"high":"SUPPORTED","low":"CONTRADICTED","medium":"UNRESOLVED"}[_band(_f1(),real_values(j)[_f1()])])),
 # CLASS: unseen restatement forms
 ("CLASS",      lambda j: (f"This session was labelled as {y_str[j]} by the collective model.","SUPPORTED")),
 ("CLASS",      lambda j: ("The session appears entirely legitimate and unremarkable.",
                           "CONTRADICTED" if y_str[j] in ATTACK else "SUPPORTED")),
 # BEHAVIOR: unseen compatibility cues
 ("BEHAVIOR",   lambda j: ("The observed pattern is characteristic of scanning.", None)),
 # CAUSALITY: unseen implicit / inverted constructions (no 'caused')
 ("CAUSALITY",  lambda j: (f"The sparse {_nm(_f1())} is what gave rise to this incident.","UNRESOLVED")),
 ("CAUSALITY",  lambda j: ("It follows from these measurements that the operator acted with hostile intent.","UNRESOLVED")),
 ("CAUSALITY",  lambda j: ("This session must therefore form part of a coordinated campaign.","UNRESOLVED")),
]
print(f"T_dev: {len(T_dev)} templates | T_test: {len(T_test)} templates")
# disjointness is by construction (different surface forms); assert no literal overlap
_d={t(0)[0] for _,t in T_dev}; _t={t(0)[0] for _,t in T_test}
assert not (_d & _t), "template leakage between dev and test"
print("T_dev intersect T_test = empty  ->  no template leakage")'''
assert hashlib.sha256(FROZEN_BANK.encode()).hexdigest() == "db273bc386d473ea76152ad17c5de4c1b19c660be6f77c9f50a1f55a126874d8"
exec(FROZEN_BANK)

In [ ]:
rng = np.random.RandomState(CFG.SEED)                    # same draw as the companion manuscript
flows = [int(x) for x in rng.choice(Xraw.shape[0], min(300, Xraw.shape[0]), replace=False)]

def build_rows(bank, setname, tid_prefix):
    rows = []
    for t, (cat, builder) in enumerate(bank):
        tid = f"{tid_prefix}{t + 1:02d}"
        for r, j in enumerate(flows):
            text, gold = builder(j)
            if gold is None:                                # behavioural template
                gold = bcon_gold(next(w for w in BCON if w in text.lower()), j)
            rows.append(dict(set=setname, stratum=setname, tid=tid, true_cat=cat, rec=j, rpos=r,
                             feat=None, text=text, gold=gold))
    return pd.DataFrame(rows)

def apply_verifiers(df):
    out = []
    for text, j in zip(df.text, df.rec):
        t0 = time.perf_counter(); v, br, cat = V0(text, j); ms = (time.perf_counter() - t0) * 1e3
        out.append((v, br, cat, ms))
    df = df.copy()
    df["V0_verdict"], df["V0_branch"], df["V0_cat"], df["V0_ms"] = zip(*out)
    v1 = [V1_from_V0(v, b, c) for v, b, c in zip(df.V0_verdict, df.V0_branch, df.V0_cat)]
    df["V1_verdict"], df["V1_branch"], df["V1_cat"] = zip(*v1)
    df["true_group"] = df.true_cat.map(group_of)
    for s in ("V0", "V1"): df[f"{s}_group"] = df[f"{s}_cat"].map(group_of)
    return df

DF_T = apply_verifiers(build_rows(T_test, "ttest", "T"))
DF_D = apply_verifiers(build_rows(T_dev, "tdev", "D"))

def core(df, s):
    ok = df[f"{s}_verdict"] == df.gold; rout = df[f"{s}_group"] == df.true_group
    fs = (df[f"{s}_verdict"] == "SUPPORTED") & (df.gold != "SUPPORTED")
    return dict(n=len(df), acc=ok.mean(), routing=rout.mean(), errors=int((~ok).sum()),
                false_supported=int(fs.sum()))
REPRO = core(DF_T, "V0"); REPRO_DEV = core(DF_D, "V0")
TARGET = dict(n=3600, acc=0.864, routing=0.833, errors=489, false_supported=218)
print("T_dev :", {k: round(v, 4) for k, v in REPRO_DEV.items()})
print("T_test:", {k: round(v, 4) for k, v in REPRO.items()})
print("paper :", TARGET)
bad = [k for k, v in TARGET.items()
       if (abs(REPRO[k] - v) > 0.0015 if isinstance(v, float) else REPRO[k] != v)]
REPRO_OK = not bad
if bad:
    msg = f"Reproduction FAILED on {bad}: artifacts or code differ from the companion manuscript's run."
    if STRICT_REPRO and not DRYRUN: raise RuntimeError(msg)
    print("WARNING:", msg, "-> nothing below is reportable.")
else:
    tag("REPRO", "T_test reproduced exactly: this is the frozen verifier on the original flows.")

## 3. T_test2 — written from the grammar, frozen before evaluation

* **Neutral N01–N21**: every grammar type (including ABSENCE, absent from T_test), syntax and
  lexical choices not in T_dev/T_test, a **feature drawn uniformly among the 25** per cell (T_test
  used only the first two), **300 flows disjoint** from T_dev/T_test.
* **Probe P01–P06**: aimed at the diagnosed patterns P1 (definitional / operator-free mention),
  P2 (*average*), P3 (*normal ranges*) and vague magnitudes. They measure the fix, not generalisation.


Gold rules use only the declared semantics (bands from `feat_q`, 2 % tolerance, the decision,
BCON); none calls `compile_claim`.

In [ ]:
# gold kinds: value(slot) | band(slot or literal) | quant(few/many) | absence | class(slot)
#             | benign | behavior(BCON name) | unres
NEUTRAL = [
 ("N01", "VALUE",      "Its {f} came out at {v}.",                                    ("value", "v")),
 ("N02", "VALUE",      "We observe {v} as the {f} of this flow.",                     ("value", "v")),
 ("N03", "VALUE",      "The recorded {f} equals {vw}.",                               ("value", "vw")),
 ("N04", "VALUE",      "For this connection, {f} stands at {vw}.",                    ("value", "vw")),
 ("N05", "COMPARISON", "In relative terms, the {f} of this flow is {b}.",             ("band", "b")),
 ("N06", "COMPARISON", "Against the corpus distribution, its {f} is {bbar}.",         ("band", "bbar")),
 ("N07", "COMPARISON", "The {f} of this flow is neither high nor low.",               ("band", "medium")),
 ("N08", "COMPARISON", "Compared with other flows, this flow's {f} looks {b}.",       ("band", "b")),
 ("N09", "QUANTIFIER", "Very few {f} are observed in this flow.",                     ("quant", "few")),
 ("N10", "QUANTIFIER", "This flow carries a great many {f}.",                         ("quant", "many")),
 ("N11", "QUANTIFIER", "Hardly any {f} appear in this traffic.",                      ("quant", "few")),
 ("N12", "ABSENCE",    "No {f} was recorded for this flow.",                          ("absence", None)),
 ("N13", "CLASS",      "The detector flagged this flow as {yhat}.",                   ("class", "yhat")),
 ("N14", "CLASS",      "This traffic was classified as {other}.",                     ("class", "other")),
 ("N15", "CLASS",      "Nothing about this session suggests an attack.",              ("benign", None)),
 ("N16", "BEHAVIOR",   "These measurements are compatible with a denial of service pattern.", ("behavior", "denial of service")),
 ("N17", "BEHAVIOR",   "The traffic profile is consistent with brute-force activity.",        ("behavior", "brute-force")),
 ("N18", "CAUSALITY",  "The high {f} led to this alert.",                             ("unres", None)),
 ("N19", "CAUSALITY",  "Because of its {f}, the flow was malicious.",                 ("unres", None)),
 ("N20", "CAUSALITY",  "The attacker clearly intended to exfiltrate data.",           ("unres", None)),
 ("N21", "CAUSALITY",  "This flow is presumably part of a larger botnet operation.",  ("unres", None)),
]
PROBE = [
 ("P01", "CAUSALITY",  "{F}: this feature indicates the {f} of the packet.",          ("unres", None)),  # P1 definitional
 ("P02", "CAUSALITY",  "The {f} is an important factor in this decision.",            ("unres", None)),  # P1 operator-free
 ("P03", "CAUSALITY",  "The {f} was taken into account by the detector.",             ("unres", None)),  # P1 operator-free
 ("P04", "VALUE",      "The average {f} of this flow is {v}.",                        ("value", "v")),   # P2 'average'
 ("P05", "CAUSALITY",  "Several values exceed normal ranges.",                        ("unres", None)),  # P3
 ("P06", "CAUSALITY",  "The {f} is unusual for this kind of traffic.",                ("unres", None)),  # vague
]
EXTERNAL = [   # (tid "E01"..., true category, text with the slots above, (gold kind, arg))
]

FREEZE_PATH = os.path.join(CFG.WORK, "ttest2_freeze.json")
payload = json.dumps(dict(neutral=NEUTRAL, probe=PROBE, external=EXTERNAL, v1_rule=V1_RULE,
                          seed2=CFG.SEED2, value_format="np.format_float_positional(precision=4)",
                          compiler_sha=hashlib.sha256(FROZEN_COMPILER.encode()).hexdigest()),
                     sort_keys=True)
SHA = hashlib.sha256(payload.encode()).hexdigest()
if os.path.exists(FREEZE_PATH):
    prev = json.load(open(FREEZE_PATH))
    if prev["sha256"] != SHA:
        raise RuntimeError(f"T_test2 or V1 changed after freezing ({prev['frozen_utc']}): "
                           "edits after evaluation void the held-out claim.")
    tag("FREEZE", f"T_test2 frozen on {prev['frozen_utc']} (sha {SHA[:16]})")
elif REPRO_OK or DRYRUN:
    json.dump(dict(sha256=SHA, frozen_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                   n_neutral=len(NEUTRAL), n_probe=len(PROBE), n_external=len(EXTERNAL),
                   payload=json.loads(payload)), open(FREEZE_PATH, "w"), indent=1)
    tag("FREEZE", f"T_test2 FROZEN now (sha {SHA[:16]}) -> {FREEZE_PATH}")

In [ ]:
rng2 = np.random.default_rng(CFG.SEED2)
pool = sorted(set(range(Xraw.shape[0])) - set(flows))
flows2 = sorted(int(x) for x in rng2.choice(pool, 300, replace=False))
def disp(f): return f.replace("_", " ")
def fmt(x):  # 4 significant digits, never scientific notation (the compiler's _num regex)
    return np.format_float_positional(float(x), precision=4, unique=False, fractional=False, trim="-")
INVB = {"low": "high", "high": "low", "medium": "medium"}

def gold_of(kind, arg, f, j, slots):
    rv = real_values(j)
    if kind == "value":
        x, w = rv[f], float(slots[arg])
        return "SUPPORTED" if abs(x - w) <= TOL * max(abs(x), 1e-6) else "CONTRADICTED"
    if kind == "band":
        claimed = slots.get(arg, arg)
        return "SUPPORTED" if _band(f, rv[f]) == claimed else "CONTRADICTED"
    if kind == "quant":
        b = _band(f, rv[f])
        if b == "medium": return "UNRESOLVED"
        return "SUPPORTED" if (b == "low") == (arg == "few") else "CONTRADICTED"
    if kind == "absence":  return "SUPPORTED" if abs(rv[f]) < 1e-9 else "CONTRADICTED"
    if kind == "class":    return "SUPPORTED" if slots[arg] == y_str[j] else "CONTRADICTED"
    if kind == "benign":   return "CONTRADICTED" if y_str[j] in ATTACK else "SUPPORTED"
    if kind == "behavior": return bcon_gold(arg, j)
    if kind == "unres":    return "UNRESOLVED"
    raise ValueError(kind)

rows = []
for stratum, temps in (("neutral", NEUTRAL), ("probe", PROBE), ("external", EXTERNAL)):
    for tid, cat, text, (kind, arg) in temps:
        for r, j in enumerate(flows2):
            f = feats[int(rng2.integers(len(feats)))]
            x = real_values(j)[f]
            others = [c for c in classes if c != y_str[j]]
            slots = dict(f=disp(f), F=disp(f).capitalize(), v=fmt(x), vw=fmt(1.5 * x + 3),
                         b=_band(f, x), bbar=INVB[_band(f, x)], yhat=y_str[j],
                         other=others[int(rng2.integers(len(others)))])
            rows.append(dict(set="ttest2", stratum=stratum, tid=tid, true_cat=cat, rec=j, rpos=r,
                             feat=f, text=text.format(**slots), gold=gold_of(kind, arg, f, j, slots)))
DF_2 = apply_verifiers(pd.DataFrame(rows))
print(DF_2.groupby("stratum").size().to_dict())
print(DF_2.groupby("stratum").gold.value_counts().unstack(fill_value=0))
DF_2.sample(8, random_state=0)[["tid", "text", "gold", "V0_cat", "V0_verdict"]]

## 4. V0 and V1: T_test (diagnostic) and T_test2 (held-out)

On new phrasings a compilation error can be a wrong **type** or the right type with a wrong
**argument** (e.g. *neither high nor low* read as *high*). The paper observed only type errors on
T_test; both are separated here. The approval invariant covers both (`ĉ ≠ c`).

In [ ]:
def cp_upper(k, n, a=0.05): return 1.0 if k >= n else float(beta.ppf(1 - a, k + 1, n - k))
def cp_lower(k, n, a=0.05): return 0.0 if k == 0 else float(beta.ppf(a, k, n - k + 1))
def cell_matrix(df, col):
    return df.pivot_table(index="tid", columns="rpos", values=col, aggfunc="mean").to_numpy(float)
def cluster_boot(M, B=CFG.B_BOOT, seed=CFG.SEED2):
    g = np.random.default_rng(seed); T, R = M.shape
    valid = (~np.isnan(M)).astype(float); Mz = np.nan_to_num(M); est = np.empty(B)
    for b in range(B):
        wt = g.multinomial(T, np.full(T, 1 / T)); wr = g.multinomial(R, np.full(R, 1 / R))
        est[b] = (wt @ Mz @ wr) / max(wt @ valid @ wr, 1)
    return np.percentile(est, [2.5, 97.5])

def report(df, s, label):
    ok = df[f"{s}_verdict"] == df.gold; typ = df[f"{s}_group"] == df.true_group
    fs = (df[f"{s}_verdict"] == "SUPPORTED") & (df.gold != "SUPPORTED")
    ex = df[f"{s}_group"].isin(EXEC_GROUPS); und = df[f"{s}_group"] == "UND"
    gU = df.gold == "UNRESOLVED"
    d = df.assign(_ok=ok.astype(float), _fs=fs.astype(float))
    nT = d.tid.nunique(); kT = int(d.groupby("tid")._fs.max().sum())
    m = dict(n=len(df), acc=float(ok.mean()), acc_ci=cluster_boot(cell_matrix(d, "_ok")).round(3).tolist(),
             routing=float(typ.mean()), verif_given_type=float(ok[typ].mean()) if typ.any() else float("nan"),
             errors=int((~ok).sum()), err_wrong_type=int((~ok & ~typ).sum()),
             err_right_type_exec_argument=int((~ok & typ & ex).sum()),
             err_right_type_constraint=int((~ok & typ & (df.true_group == "BEH")).sum()),
             err_right_type_abstain=int((~ok & typ & (df.true_group == "UND")).sum()),
             false_supported=int(fs.sum()), fs_rate=float(fs.mean()),
             fs_ci_cluster=cluster_boot(cell_matrix(d, "_fs")).round(3).tolist(),
             fs_from_exec=int((fs & ex).sum()), fs_wrong_type=int((fs & ~typ).sum()),
             fs_right_type_argument=int((fs & typ).sum()),
             fs_bound_claims=round(cp_upper(int(fs.sum()), len(df)), 4),
             templates_with_false_sup=f"{kT}/{nT}", fs_bound_templates=round(cp_upper(kT, nT), 4),
             templates_perfect=f"{int((d.groupby('tid')._ok.mean() == 1).sum())}/{nT}",
             abst_prec=float((gU & und).sum() / max(und.sum(), 1)),
             abst_rec=float((gU & und).sum() / max(gU.sum(), 1)))
    m["assumption2_holds"] = m["fs_from_exec"] == m["false_supported"]
    print(f"\n=== {label} | {s} ===")
    for k, v in m.items(): print(f"  {k:30s} {round(v, 4) if isinstance(v, float) else v}")
    return m

RES = {"meta": dict(dryrun=DRYRUN, repro_ok=REPRO_OK, reproduction=REPRO, freeze_sha256=SHA,
                    compiler_sha256=hashlib.sha256(FROZEN_COMPILER.encode()).hexdigest(), v1_rule=V1_RULE,
                    flows_ttest2_disjoint=True, run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())}
RES["ttest_V0"] = report(DF_T, "V0", "T_test (paper)")
RES["ttest_V1"] = report(DF_T, "V1", "T_test - DIAGNOSTIC (rule motivated here)")
for st in ("neutral", "probe", "external"):
    sub = DF_2[DF_2.stratum == st]
    if len(sub):
        for s in ("V0", "V1"): RES[f"ttest2_{st}_{s}"] = report(sub, s, f"T_test2 {st}")

In [ ]:
# Per-template table (appendix) and compiler confusion (Figure 2 analogue)
def per_template(df):
    g = df.groupby(["stratum", "tid", "true_cat"])
    return pd.DataFrame({
        "V0_acc":   g.apply(lambda d: (d.V0_verdict == d.gold).mean()),
        "V1_acc":   g.apply(lambda d: (d.V1_verdict == d.gold).mean()),
        "V0_route": g.apply(lambda d: (d.V0_group == d.true_group).mean()),
        "V0_FS":    g.apply(lambda d: int(((d.V0_verdict == "SUPPORTED") & (d.gold != "SUPPORTED")).sum())),
        "V1_FS":    g.apply(lambda d: int(((d.V1_verdict == "SUPPORTED") & (d.gold != "SUPPORTED")).sum())),
        "V0_main_cat": g.apply(lambda d: d.V0_cat.value_counts().index[0]),
        "example":  g.apply(lambda d: d.text.iloc[0][:70]),
    }).round(3)
PT2 = per_template(DF_2)
sub = DF_2[DF_2.stratum == "neutral"]
for s in ("V0", "V1"):
    print(f"\n{s}: true group (rows) x compiled group (cols), T_test2 neutral")
    print(pd.crosstab(sub.true_group, sub[f"{s}_group"]))
PT2

## 5. Baselines on the same claims — resumable

Premise = the serialised evidence of the companion manuscript's NLI baseline, extended
to all 25 features: `The <f> is <value> (<band> range).` … `The classifier predicts <ŷ>.`
Variant *own* keeps only the claim's feature.
MiniCheck is binary, so the common comparison is the **SUPPORTED decision** (precision, recall on true
claims, false acceptances), with exact McNemar on discordant pairs and a two-way cluster bootstrap
on the paired difference. NLI keeps its 3-way output (4-way accuracy without behavioural claims).

**Why this section is built to survive interruptions.** MiniCheck scores one claim at a time
(≈ 13 claims/s for RoBERTa-L, ≈ 8/s for Flan-T5-L on a T4), so the baselines take ≈ 45 min, during
which Colab may disconnect (idle tab, usage limits, front-end overload by progress bars).
Hence: progress bars are silenced; every prediction is appended to a cache file on Drive in chunks of
500 claims; one model is in memory at a time; a re-run skips everything already cached (including the
NLI and MiniCheck-RoBERTa predictions of a previous interrupted run, imported from
`checkpoint_ttest2.csv`). **If the session dies: reconnect and Run all again** — sections 0–4 take a
few minutes and are deterministic (T_test2 stays frozen), and section 5 resumes where it stopped.

In [ ]:
# === 5a. Resumable baseline engine ==============================================
import gc
os.environ["TQDM_DISABLE"] = "1"            # tqdm >= 4.66 honours this; patched again below
def premise_full(j):
    rv = real_values(j)
    return " ".join(f"The {f.replace('_', ' ')} is {round(rv[f], 3)} ({_band(f, rv[f])} range)." for f in feats) \
           + f" The classifier predicts {y_str[j]}."
def premise_own(j, f):
    if not isinstance(f, str): return premise_full(j)
    rv = real_values(j)
    return f"The {f.replace('_', ' ')} is {round(rv[f], 3)} ({_band(f, rv[f])} range). The classifier predicts {y_str[j]}."

NLI_LABELS = ["contradiction", "entailment", "neutral"]              # model card order
NLI2V = {"entailment": "SUPPORTED", "contradiction": "CONTRADICTED", "neutral": "UNRESOLVED"}
CACHE = os.path.join(CFG.WORK, "baseline_cache"); os.makedirs(CACHE, exist_ok=True)
CHUNK = 500
SETS = {"ttest2": DF_2, "ttest": DF_T}
for _k, _d in SETS.items():
    _d["key"] = _d.tid + "|" + _d.rec.astype(str)
    assert _d.key.is_unique, f"duplicate (template, flow) keys in {_k}"

def _cache_path(setname, method): return os.path.join(CACHE, f"{setname}__{method}.csv")
def _cached(setname, method):
    p = _cache_path(setname, method)
    if not os.path.exists(p): return {}
    c = pd.read_csv(p, dtype={"key": str, "pred": str}).drop_duplicates("key", keep="last")
    return dict(zip(c.key, c.pred))
def _append(setname, method, keys, preds):
    p = _cache_path(setname, method); new = not os.path.exists(p)
    pd.DataFrame({"key": keys, "pred": preds}).to_csv(p, mode="a", header=new, index=False)

# Import predictions already computed by an interrupted run (old checkpoint files), after
# checking that rows match exactly (same template, flow and text).
for setname, df in SETS.items():
    ck = os.path.join(CFG.WORK, f"checkpoint_{setname}.csv")
    if not os.path.exists(ck): continue
    old = pd.read_csv(ck)
    old["key"] = old.tid + "|" + old.rec.astype(str)
    m = df[["key", "text"]].merge(old, on="key", suffixes=("", "_old"))
    if len(m) != len(df) or not (m.text == m.text_old).all():
        tag("RESUME", f"{ck} does not match the current claims -> ignored"); continue
    for col in ["NLI_full", "NLI_own"] + [f"MiniCheck-{x}" for x in CFG.MINICHECK]:
        if col in m.columns and m[col].notna().all() and len(_cached(setname, col)) < len(df):
            _append(setname, col, m.key.tolist(), m[col].astype(str).tolist())
            tag("RESUME", f"imported {col} for {setname} from the interrupted run ({len(m)} claims)")

def run_method(method, predict_fn, premise_fn):
    # predict_fn(premises, claims) -> list of labels; resumable, chunked, silent
    for setname in CFG.BASELINE_SETS:
        df = SETS[setname]; done = _cached(setname, method)
        todo = df[~df.key.isin(done.keys())]
        if todo.empty:
            tag("CACHE", f"{method} on {setname}: complete ({len(df)} claims)"); continue
        t0 = time.time(); n0 = len(todo)
        for a in range(0, len(todo), CHUNK):
            part = todo.iloc[a:a + CHUNK]
            prem = [premise_fn(j, f) for j, f in zip(part.rec, part.feat)]
            _append(setname, method, part.key.tolist(), predict_fn(prem, part.text.tolist()))
            k = a + len(part); el = time.time() - t0
            print(f"  {method} | {setname}: {len(df) - n0 + k}/{len(df)}  "
                  f"({el / 60:.1f} min, ~{el / k * (n0 - k) / 60:.1f} min left)", flush=True)

def timed(fn, n=200):
    sub = DF_2[DF_2.stratum == "neutral"].sample(n, random_state=CFG.SEED); ts = []
    for j, t in zip(sub.rec, sub.text):
        t0 = time.perf_counter(); fn([premise_full(j)], [t]); ts.append(time.perf_counter() - t0)
    return 1e3 * float(np.median(ts))
COST_PATH = os.path.join(CACHE, "cost_ms.json")
COST = json.load(open(COST_PATH)) if os.path.exists(COST_PATH) else {}
COST["V0/V1 routed, CPU"] = float(np.median(DF_2[DF_2.stratum == "neutral"].V0_ms))
def save_cost(): json.dump(COST, open(COST_PATH, "w"), indent=1)

if not DRYRUN:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU: Runtime > Change runtime type > T4 GPU, then Run all "
                           "(MiniCheck-Flan-T5 on CPU would take many hours).")
    tag("GPU", torch.cuda.get_device_name(0))

In [ ]:
# === 5b. NLI cross-encoder baseline, full and own-feature premises ===
def _need(method):
    return any(len(_cached(s, method)) < len(SETS[s]) for s in CFG.BASELINE_SETS)

if DRYRUN:
    _g = np.random.default_rng(1)
    _nli_fn = lambda P, C: list(_g.choice(list(NLI2V.values()), len(C)))
else:
    from sentence_transformers import CrossEncoder
    _nli_model = CrossEncoder(CFG.NLI_MODEL, device="cuda")
    def _nli_fn(P, C):
        p = np.asarray(_nli_model.predict(list(zip(P, C)), apply_softmax=True, batch_size=64,
                                          show_progress_bar=False))
        return [NLI2V[NLI_LABELS[int(k)]] for k in p.argmax(1)]
if _need("NLI_full"): run_method("NLI_full", _nli_fn, lambda j, f: premise_full(j))
else: tag("CACHE", "NLI_full complete")
if _need("NLI_own"):  run_method("NLI_own", _nli_fn, premise_own)
else: tag("CACHE", "NLI_own complete")
if "NLI 142M" not in COST: COST["NLI 142M"] = timed(_nli_fn); save_cost()
if not DRYRUN:
    del _nli_model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === 5c. MiniCheck, one model at a time (RoBERTa-L ~10 min, Flan-T5-L ~25 min on a T4) ===
for m in CFG.MINICHECK:
    method = f"MiniCheck-{m}"
    if not _need(method) and method in COST:
        tag("CACHE", f"{method} complete"); continue
    if DRYRUN:
        _g = np.random.default_rng(len(m))
        fn = lambda P, C: list(_g.choice(["SUPPORTED", "UNSUPPORTED"], len(C)))
    else:
        import minicheck.inference as _mci
        _mci.tqdm = lambda it, *a, **k: it                 # silence per-claim progress bars
        from minicheck.minicheck import MiniCheck
        sc = MiniCheck(model_name=m, cache_dir="./ckpts")
        def fn(P, C, sc=sc):
            lab, _, _, _ = sc.score(docs=list(P), claims=list(C))
            return ["SUPPORTED" if int(l) == 1 else "UNSUPPORTED" for l in lab]
    tag("MINICHECK", f"{method}: start")
    run_method(method, fn, lambda j, f: premise_full(j))
    if method not in COST: COST[method] = timed(fn); save_cost()
    if not DRYRUN:
        del sc, fn; gc.collect(); torch.cuda.empty_cache()
    tag("MINICHECK", f"{method}: done")
print({k: round(v, 3) for k, v in COST.items()}, "ms / claim (median, single-claim calls)")

In [ ]:
# === 5d. Attach cached predictions to the claim tables ==========================
BASE_COLS_BY_SET = {}
for setname in CFG.BASELINE_SETS:
    df = SETS[setname]; cols = []
    for method in ["NLI_full", "NLI_own"] + [f"MiniCheck-{m}" for m in CFG.MINICHECK]:
        c = _cached(setname, method)
        if len(c) >= len(df) and df.key.isin(c.keys()).all():
            df[method] = df.key.map(c); cols.append(method)
        else:
            tag("MISSING", f"{method} on {setname}: {len(c)}/{len(df)} cached -> left out of the tables")
    BASE_COLS_BY_SET[setname] = cols
DF_2, DF_T = SETS["ttest2"], SETS["ttest"]
print(BASE_COLS_BY_SET)

In [ ]:
def binary_view(df, col):
    pos = df.gold == "SUPPORTED"; pred = df[col] == "SUPPORTED"
    tp, fp = int((pos & pred).sum()), int((~pos & pred).sum())
    return dict(n=len(df), true_claims=int(pos.sum()), confirmed=tp, recall_true=tp / max(pos.sum(), 1),
                precision=tp / max(tp + fp, 1), false_accept=fp, false_accept_rate=fp / max((~pos).sum(), 1),
                sup_decision_acc=float((pos == pred).mean()))
def paired(df, a, b):
    ca = (df[a] == "SUPPORTED") == (df.gold == "SUPPORTED"); cb = (df[b] == "SUPPORTED") == (df.gold == "SUPPORTED")
    n10, n01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    ci = cluster_boot(cell_matrix(df.assign(_d=ca.astype(float) - cb.astype(float)), "_d"))
    return dict(a=a, b=b, a_right_b_wrong=n10, a_wrong_b_right=n01, mcnemar_p=float(p),
                diff=float(ca.mean() - cb.mean()), diff_ci_cluster=ci.round(3).tolist())

BIN, PAIR = {}, []
for setname in CFG.BASELINE_SETS:
    df = SETS[setname]; cols = BASE_COLS_BY_SET[setname]
    base = df[df.stratum.isin(["neutral", "ttest"])]
    for sname, sub in {"all": base, "computable": base[base.true_group.isin(EXEC_GROUPS)]}.items():
        for col in ["V0_verdict", "V1_verdict"] + cols:
            BIN[f"{setname}|{sname}|{col}"] = binary_view(sub, col)
        for col in cols:
            for s in ("V0_verdict", "V1_verdict"):
                PAIR.append(dict(set=setname, subset=sname, **paired(sub, s, col)))
    nb = base[base.true_group != "BEH"]
    for col in ["V0_verdict", "V1_verdict"] + [c for c in cols if c.startswith("NLI")]:
        BIN[f"{setname}|no_behaviour|4way|{col}"] = dict(acc=float((nb[col] == nb.gold).mean()), n=len(nb))
TAB_BIN = pd.DataFrame(BIN).T; TAB_PAIR = pd.DataFrame(PAIR)
if "ttest" in CFG.BASELINE_SETS and "NLI_own" in BASE_COLS_BY_SET.get("ttest", []):
    v = BIN["ttest|all|NLI_own"]
    print(f"Sanity vs paper (T_test, NLI): SUPPORTED returned {int((DF_T.NLI_own == 'SUPPORTED').sum())}"
          f" / {len(DF_T)}, true claims confirmed {v['confirmed']}/{v['true_claims']} "
          "(paper: 1 / 3600 and 1 / 1,253; premise format may differ slightly)")
display(TAB_BIN.round(3)); display(TAB_PAIR.round(4))

## 6. Real rationales under V0 and V1 (annotation-free)

Earlier free-form explanations, scored sentence by sentence (`split_sentences`, `is_meta`, decision from the CSV).
V0 must give 41.0 % for TinyLlama. The Qwen2.5 file is found on Drive automatically (name contains
*qwen*, columns detected; a numeric decision is mapped to its class name); its V0 rate is compared with
the companion manuscript's 54.0 %. Under V1 the operator-free pattern P1 can no longer produce a
refutation, so V1's raw rate is a structural, annotation-free re-estimate of the bracket's upper side.

In [ ]:
def _find_csv():   # identical preference order to plan_verify_generate_audit
    import glob
    for c in [os.path.join(CFG.OUTPUT_DIR, f"soc_eval_grounded_{CFG.DATASET}.csv"),
              os.path.join(CFG.OUTPUT_DIR, f"soc_eval_full_{CFG.DATASET}.csv")]:
        if os.path.exists(c): return c
    g = glob.glob(CFG.DRIVE_ROOT + "/**/soc_eval_*" + CFG.DATASET + "*.csv", recursive=True)
    return g[0] if g else None

IDX_COLS  = ["idx", "flow_idx", "flow_id", "index", "j", "record", "row"]
TEXT_COLS = ["rationale_grounded", "rationale", "explanation", "text", "response", "output",
             "generation", "generated", "completion", "answer"]

def _columns(df):
    low = {c.lower(): c for c in df.columns}
    ic = next((low[c] for c in IDX_COLS if c in low), None)
    tc = next((low[c] for c in TEXT_COLS if c in low), None)
    if tc is None:   # fall back to the longest string column
        obj = [c for c in df.columns if df[c].dtype == object]
        if obj: tc = max(obj, key=lambda c: df[c].astype(str).str.len().mean())
    pc = low.get("pred") or low.get("prediction") or low.get("yhat") or low.get("decision")
    return ic, tc, pc

def _find_qwen():
    import glob
    if CFG.QWEN_CSV not in (None, "auto"):
        return CFG.QWEN_CSV if os.path.exists(CFG.QWEN_CSV) else None
    if CFG.QWEN_CSV is None: return None
    pat = "/**/*[Qq][Ww][Ee][Nn]*.csv"
    roots = [os.path.dirname(CFG.OUTPUT_DIR.rstrip("/")), CFG.MODELS_ROOT]   # project folders first
    cands = sorted({p for r in roots if os.path.isdir(r) for p in glob.glob(r + pat, recursive=True)})
    if not cands:
        cands = sorted(glob.glob(CFG.DRIVE_ROOT + pat, recursive=True))     # then the whole Drive
    good = []
    for p in cands:
        try:
            d = pd.read_csv(p, nrows=5); ic, tc, _ = _columns(d)
            if ic is not None and tc is not None:
                score = (CFG.DATASET.lower() in p.lower()) * 2 + ("rational" in p.lower()) + ("explain" in p.lower())
                good.append((score, os.path.getmtime(p), p))
        except Exception:
            pass
    tag("QWEN", f"{len(cands)} CSV named *qwen* found, {len(good)} with index + text columns")
    if not good: return None
    best = sorted(good, reverse=True)[0][2]
    if len(good) > 1:
        tag("QWEN", "candidates: " + " | ".join(os.path.basename(g[2]) for g in sorted(good, reverse=True)))
    return best

def _pred_name(p):
    # Qwen files may store the decision as a class index; map it to the class name.
    if p is None or (isinstance(p, float) and np.isnan(p)): return None
    if isinstance(p, (int, np.integer)) or (isinstance(p, str) and p.strip().isdigit()):
        k = int(p); return classes[k] if 0 <= k < len(classes) else None
    return str(p)

def raw_rates(path, max_rows=300, exact_paper_loop=True):
    df = pd.read_csv(path)
    if exact_paper_loop:   # TinyLlama: byte-for-byte the companion manuscript's scoring loop
        ic, pc = "idx", "pred"
        col = "rationale_grounded" if "rationale_grounded" in df.columns else "rationale"
        getp = lambda r: r.get("pred")
    else:                  # Qwen: column names detected, decision normalised to a class name
        ic, col, pc = _columns(df)
        getp = (lambda r: _pred_name(r[pc])) if pc else (lambda r: None)
    n = h0 = h1 = 0; cats = {}
    for _, r in df.head(max_rows).iterrows():
        j = int(r[ic])
        if not (0 <= j < Xraw.shape[0]): continue
        n += 1; a0 = a1 = False
        for s in split_sentences(str(r[col])):
            if is_meta(s): continue
            v, br, cat = verify(s, j, getp(r))
            cats[cat] = cats.get(cat, 0) + 1
            a0 |= (br == "EXEC" and v == "CONTRADICTED")
            v1, b1, _ = V1_from_V0(v, br, cat)
            a1 |= (b1 == "EXEC" and v1 == "CONTRADICTED")
        h0 += a0; h1 += a1
    n = max(n, 1)
    return dict(file=path, columns=dict(index=ic, text=col, decision=pc), rationales=n,
                raw_V0=h0 / n, raw_V0_count=h0, raw_V1=h1 / n, raw_V1_count=h1,
                raw_V1_ci95=[round(cp_lower(h1, n, .025), 3), round(cp_upper(h1, n, .025), 3)],
                compiled_categories=cats)

REAL = {}
p = _find_csv()
if p: REAL["TinyLlama-1.1B"] = raw_rates(p, exact_paper_loop=True)
else: tag("REAL", "TinyLlama rationale CSV not found")
q = _find_qwen()
if q: REAL["Qwen2.5-1.5B"] = raw_rates(q, exact_paper_loop=False)
else: tag("REAL", "no Qwen rationale CSV found on Drive -> Qwen line skipped (TinyLlama only)")
print(json.dumps(REAL, indent=1) if REAL else "no rationale file found")
if "TinyLlama-1.1B" in REAL and not DRYRUN:
    print("TinyLlama V0 reproduces the companion manuscript's 41.0 %:", abs(REAL["TinyLlama-1.1B"]["raw_V0"] - 0.410) < 0.002)
if "Qwen2.5-1.5B" in REAL and not DRYRUN:
    print("Qwen V0 vs companion manuscript's 54.0 %:", round(REAL["Qwen2.5-1.5B"]["raw_V0"], 3),
          "(check the file name above if it differs: another Qwen CSV may have been picked)")

## 7. Save, LaTeX rows, and what the results license

In [ ]:
RES.update(baseline_binary=TAB_BIN.to_dict(orient="index"), baseline_paired=PAIR, cost_ms=COST,
           real_rationales=REAL, per_template_ttest2=PT2.reset_index().to_dict(orient="records"))
json.dump(RES, open(os.path.join(CFG.WORK, "ttest2_results.json"), "w"), indent=1, default=float)
pd.concat([DF_D, DF_T, DF_2]).to_csv(os.path.join(CFG.WORK, "ttest2_claims.csv"), index=False)

def row(name, m):
    return (f"{name} & {m['acc']:.3f} [{m['acc_ci'][0]:.2f}, {m['acc_ci'][1]:.2f}] & {m['routing']:.3f} & "
            f"{m['verif_given_type']:.3f} & {m['false_supported']} ({100 * m['fs_rate']:.1f}\\%) & {m['templates_perfect']} \\\\")
L = [r"% verifier & acc [two-way cluster 95% CI] & routing & verif|type & false SUP & templates perfect"]
for k, name in [("ttest_V0", "V0, $T_{test}$"), ("ttest_V1", "V1, $T_{test}$ (diagnostic)"),
                ("ttest2_neutral_V0", "V0, $T_{test2}$"), ("ttest2_neutral_V1", "V1, $T_{test2}$"),
                ("ttest2_probe_V0", "V0, probes"), ("ttest2_probe_V1", "V1, probes"),
                ("ttest2_external_V0", "V0, external"), ("ttest2_external_V1", "V1, external")]:
    if k in RES: L.append(row(name, RES[k]))
L.append(r"% SUPPORTED decision, T_test2 neutral, computable claims: confirmed/true & precision & false accept")
for k, v in BIN.items():
    if k.startswith("ttest2|computable|"):
        L.append(f"{k.split('|')[-1]} & {v['confirmed']}/{v['true_claims']} & {v['precision']:.3f} & {v['false_accept']} \\\\")
open(os.path.join(CFG.WORK, "ttest2_tables.tex"), "w").write("\n".join(L))
print("\n".join(L)); print("\nsaved in", CFG.WORK)

In [ ]:
if DRYRUN or not REPRO_OK:
    print("Dry-run or failed reproduction: NOTHING here is reportable.")
else:
    a, b = RES["ttest2_neutral_V0"], RES["ttest2_neutral_V1"]
    print(f"T_test2 neutral: {b['n']} claims, 21 templates x 300 new flows")
    print(f"  end-to-end   V0 {a['acc']:.3f} {a['acc_ci']}  ->  V1 {b['acc']:.3f} {b['acc_ci']}")
    print(f"  false SUP    V0 {a['false_supported']}  ->  V1 {b['false_supported']}  "
          f"(95% upper: {b['fs_bound_claims']} claim-level, {b['fs_bound_templates']} template-level)")
    print(f"  of V1's false SUP: wrong type {b['fs_wrong_type']}, right type + wrong argument {b['fs_right_type_argument']}")
    print(f"  all false SUP emitted by EXEC (Assumption 2 / Prop. 3): {b['assumption2_holds']}")
    print(f"  verification | correct type: {b['verif_given_type']:.3f}  "
          f"(argument errors: {b['err_right_type_exec_argument']})")
    for p in PAIR:
        if p["set"] == "ttest2" and p["subset"] == "computable" and p["a"] == "V1_verdict":
            print(f"  V1 vs {p['b']:28s} discordant {p['a_right_b_wrong']}/{p['a_wrong_b_right']}  "
                  f"p={p['mcnemar_p']:.1e}  diff {p['diff']:+.3f} {p['diff_ci_cluster']}")
    if "TinyLlama-1.1B" in REAL:
        r = REAL["TinyLlama-1.1B"]; print(f"  TinyLlama rationales: raw {r['raw_V0']:.3f} (V0) -> {r['raw_V1']:.3f} (V1)")
    if "Qwen2.5-1.5B" in REAL:
        r = REAL["Qwen2.5-1.5B"]; print(f"  Qwen2.5 rationales:   raw {r['raw_V0']:.3f} (V0) -> {r['raw_V1']:.3f} (V1)")